# Homework 5: Grover's search in Qiskit
ECSE 4969/6969 Quantum Information Processing, Fall 2026

This notebook is for **Problems 3 and 4** of Homework 5. Open it in Colab, save a copy to your Drive, and run the cells **in order**, top to bottom (click a cell, then press the play button at its left, or Shift+Enter).

Conventions are the same as in Lecture 12: qubit 0 is A (the first bit of a position), qubit 1 is B, and qubit 2 is C. Qiskit prints measured strings with qubit 0 on the **right**, so a two-qubit position AB is printed as BA.

You do not design circuits or write new code. For your PDF, include the counts from Cell 5, the three results from Cell 7 and the amplitude arrays from Cell 8. Installation output and repeated runs are not needed. The cell numbers below are the headings "Cell 1" to "Cell 8", not Colab's execution counters.

## Cell 1. Install Qiskit into this session
Run this cell whenever you connect to a new Colab runtime. It takes about a minute.

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

## Cell 2. Check that it works
This should print a version number.

In [ ]:
import qiskit
print(qiskit.__version__)
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
sim = AerSimulator()

## Cell 3. The fixed amplification block D for two qubits
This is the block from Lecture 12. It equals D = 2|all_2><all_2| - I exactly. Run this cell once; later cells call `add_D(qc)`.

In [ ]:
def add_D(qc):
    """Append the fixed two-qubit amplification block D to the circuit qc."""
    qc.h([0, 1])
    qc.z([0, 1])
    qc.cz(0, 1)   # part of D, not an oracle call
    qc.h([0, 1])

## Cell 4. Two marked positions
In lecture the oracle used an extra qubit in |−⟩; it left that qubit unchanged and gave the matching argument term a minus sign. Here that same sign change is applied directly to the two argument qubits, so the extra qubit is omitted (Lecture 12, optional slides 31–37). CZ changes the sign of |11⟩ only, and X interchanges |0⟩ and |1⟩.

The first circuit marks AB = 11 with a CZ. The second marks AB = 01 by putting an X on qubit 0 (A) before and after the CZ. Run the cell and check that the printed strings match Lecture 12.

In [ ]:
g11 = QuantumCircuit(2)
g11.h([0, 1])
g11.cz(0, 1)                    # oracle: AB = 11
add_D(g11)
g11.measure_all()
print("marked AB = 11:", sim.run(transpile(g11, sim), shots=1024).result().get_counts())

g01 = QuantumCircuit(2)
g01.h([0, 1])
g01.x(0); g01.cz(0, 1); g01.x(0)  # oracle: AB = 01
add_D(g01)
g01.measure_all()
print("marked AB = 01:", sim.run(transpile(g01, sim), shots=1024).result().get_counts())

## Cell 5. Problem 3.3: two more oracles
The oracle for AB = 00 puts X on both qubits before and after the CZ. The oracle for AB = 10 puts X on qubit 1 (B) before and after the CZ. The D block is unchanged. Before running, write down the string you expect Qiskit to print for each circuit.

In [ ]:
g00 = QuantumCircuit(2)
g00.h([0, 1])
g00.x([0, 1]); g00.cz(0, 1); g00.x([0, 1])  # oracle: AB = 00
add_D(g00)
g00.measure_all()
print("marked AB = 00:", sim.run(transpile(g00, sim), shots=1024).result().get_counts())

g10 = QuantumCircuit(2)
g10.h([0, 1])
g10.x(1); g10.cz(0, 1); g10.x(1)  # oracle: AB = 10
add_D(g10)
g10.measure_all()
print("marked AB = 10:", sim.run(transpile(g10, sim), shots=1024).result().get_counts())

## Cell 6. Problem 4: an eight-entry search
Three argument qubits, N = 8, with the marked position ABC = 101 (the instance of Problem 1). The function below prepares the qubits and applies m repetitions of the oracle followed by an amplification block. You do not need to change it. Run this cell once.

The doubly controlled Z (CCZ) is one joint three-qubit gate, built here as H on qubit 2, a Toffoli gate (`ccx`), and H on qubit 2. The amplification block is: H on each qubit; X on each qubit; one CCZ; X on each qubit; H on each qubit. Problem 4.3 asks how it relates to D.

In [ ]:
def add_ccz(qc):
    qc.h(2); qc.ccx(0, 1, 2); qc.h(2)

def grover8(m):
    qc = QuantumCircuit(3)
    qc.h([0, 1, 2])                          # prepare
    for _ in range(m):
        qc.x(1); add_ccz(qc); qc.x(1)        # oracle: marks ABC = 101
        qc.h([0, 1, 2]); qc.x([0, 1, 2])     # amplification block
        add_ccz(qc)
        qc.x([0, 1, 2]); qc.h([0, 1, 2])
    return qc

grover8(1).draw("mpl")

## Cell 7. Problem 4.1: run it and count
Each value of m is a separate circuit, run for 4,096 fresh shots. Predict the fraction of `101` for each m before you run the cell.

In [ ]:
shots = 4096
for m in [1, 2, 3]:
    qc = grover8(m).measure_all(inplace=False)
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    k = counts.get("101", 0)
    print(f"m = {m}: '101' appeared {k} times out of {shots}, fraction {k/shots:.4f}")

## Cell 8. Problem 4.3: the amplitudes after m repetitions
This is a simulation, not a measurement. It prints the eight amplitudes for Qiskit's labels CBA = 000, 001, ..., 111, in that order. The marked position 101 reads the same in either order. Only real parts are printed, rounded to three decimals, because every amplitude here is real.

In [ ]:
for m in [1, 2]:
    amps = Statevector.from_instruction(grover8(m)).data.real.round(3) + 0.0
    print(f"m = {m}:", amps)